# Dataset refinement and training notebook

This notebook cleans raw datasets, normalizes labels, creates language-aware splits, and trains a multilingual baseline for fake-news detection.

Recommended Kaggle dataset: `Multilingual Fake News Detection Dataset for mBERT`


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path('..').resolve()
RAW = ROOT / 'data' / 'raw'
REFINED = ROOT / 'data' / 'refined'
MODELS = ROOT / 'models'
RAW.mkdir(parents=True, exist_ok=True)
REFINED.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

print('Folders ready:', RAW, REFINED, MODELS)


In [ ]:
# Download or import dataset
# Recommended Kaggle dataset: `maulishkasrivastava/multilingual-fake-news-mbert`
# You can also use local CSV files saved into data/raw/

raw_files = list(RAW.glob('*.csv')) + list(RAW.glob('*.parquet'))
print('Available raw files:', raw_files)
if not raw_files:
    raise FileNotFoundError('Place a CSV or Parquet dataset in data/raw/')

df = pd.read_csv(raw_files[0])
print(df.head())
print(df.columns.tolist())


In [ ]:
import re
import html

def normalize_label(value):
    v = str(value).strip().lower()
    if v in {'real', 'true', 'reliable', '1', '1.0'} or v.startswith('real'):
        return 'REAL'
    if v in {'fake', 'false', 'unreliable', '0', '0.0'} or v.startswith('fake'):
        return 'FAKE'
    raise ValueError(f'Unsupported label: {value!r}')

def clean_text(value):
    s = html.unescape(str(value or ''))
    s = s.lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
    s = re.sub(r'[^a-z0-9\s\u0900-\u097f]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

required = {'text', 'label'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing columns: {missing}')

df = df[['text', 'label']].dropna().copy()
df['text'] = df['text'].map(clean_text)
df['label'] = df['label'].map(normalize_label)
df = df[df['text'].str.len() > 0].drop_duplicates().reset_index(drop=True)
print(df.head())
print('Rows after cleaning:', len(df))


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

X = df['text']
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=2)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=2000, class_weight='balanced')
model.fit(X_train_vec, y_train)
preds = model.predict(X_test_vec)

print('Accuracy:', accuracy_score(y_test, preds))
print('Precision:', precision_score(y_test, preds, average='macro', zero_division=0))
print('Recall:', recall_score(y_test, preds, average='macro', zero_division=0))
print('F1:', f1_score(y_test, preds, average='macro', zero_division=0))
print(classification_report(y_test, preds, zero_division=0))
print(confusion_matrix(y_test, preds, labels=['FAKE', 'REAL']))

refined_path = REFINED / 'refined_dataset.csv'
df.to_csv(refined_path, index=False)
print('Saved refined dataset to:', refined_path)

import joblib
joblib.dump(vectorizer, MODELS / 'baseline' / 'tfidf_vectorizer.joblib')
joblib.dump(model, MODELS / 'baseline' / 'logreg_model.joblib')
print('Saved baseline model to:', MODELS / 'baseline')


## Next steps
- Add a multilingual transformer (XLM-R / DistilBERT).
- Evaluate per-language metrics for English, Hindi, Hinglish.
- Save checkpoints into `models/transformer/`.
- Use the refined dataset for final training and validation.
